# Chapter 7 — Fine-tuning to follow instructions

Chapter 6 adapted pretrained GPT-2 representations to a fixed two-label classifier. Instruction fine-tuning keeps the language-model
output head and instead teaches the model to continue structured requests with useful natural-language responses.

The underlying objective remains next-token prediction. The difference is the supervised corpus: each example pairs an instruction and
optional task input with a desired response. This chapter prepares those examples, batches their token IDs, and fine-tunes the pretrained
model to reproduce the response pattern.

## 7.1 Downloading and validating instruction data

The book provides 1,100 supervised examples as JSON. Cache the file locally so notebook reruns avoid unnecessary network work, then
validate its external schema before the rest of the notebook relies on typed fields.

Each entry must contain three strings: `instruction`, `input`, and `output`. Validation catches truncated files or unexpected source
changes near the acquisition boundary.

In [5]:
import json
import urllib.request
from pathlib import Path
from typing import TypedDict, cast


class InstructionExample(TypedDict):
    """Schema for one supervised instruction-response example."""

    instruction: str
    input: str
    output: str


def download_and_load_file(
    file_path: str | Path,
    url: str,
) -> list[InstructionExample]:
    """Download and validate the instruction dataset when absent locally.

    Args:
        file_path: Local JSON path used as a persistent cache.
        url: Remote URL used only when the cache does not exist.

    Returns:
        Validated instruction examples with `instruction`, `input`, and
        `output` string fields.

    Raises:
        urllib.error.URLError: If the dataset download fails.
        UnicodeDecodeError: If downloaded bytes are not valid UTF-8.
        OSError: If reading or writing the local file fails.
        json.JSONDecodeError: If the local file is not valid JSON.
        ValueError: If the decoded JSON does not match the expected schema.
    """
    path = Path(file_path)
    if not path.exists():
        # The dataset is small enough to decode and write in one operation.
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        path.write_text(text_data, encoding="utf-8")

    raw_data = json.loads(path.read_text(encoding="utf-8"))
    required_fields = ("instruction", "input", "output")
    if not isinstance(raw_data, list) or not all(
        isinstance(entry, dict)
        and all(isinstance(entry.get(field), str) for field in required_fields)
        for entry in raw_data
    ):
        raise ValueError("Instruction data does not match the expected schema")
    # Runtime validation above makes this third-party JSON boundary safe to type.
    return cast(list[InstructionExample], raw_data)


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


## 7.2 Inspecting the instruction schema

An instruction describes the operation, optional input supplies data needed by that operation, and output contains the desired answer.
The spelling example uses all three fields.

In [6]:
# This example contains all three schema fields, including optional input.
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


### Instructions without additional input

Some tasks are fully specified by their instruction. Their `input` field is an empty string rather than a missing key, allowing every
entry to preserve one predictable schema.

In [7]:
# An empty input means the instruction is self-contained.
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


## 7.3 Formatting examples in Alpaca style

A consistent textual template tells the model which part describes the task, which part supplies optional data, and where its response
should begin. `format_input` deliberately omits the reference `output`; that answer is appended separately as the continuation the model
must learn to generate.

```text
Below is an instruction ...

### Instruction:
{instruction}

### Input:                 included only when nonempty
{input}

### Response:
{output}
```

In [8]:
def format_input(entry: InstructionExample) -> str:
    """Format an instruction and optional input without its answer.

    Args:
        entry: One validated instruction-response example.

    Returns:
        Alpaca-style model input ending before the response header.
    """
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    # Omit the complete section when no additional task input is required.
    input_text = (
        f"\n\n### Input:\n{entry['input']}"
        if entry["input"]
        else ""
    )
    return instruction_text + input_text

### Formatting an example with task input

The misspelled word appears under `### Input:`, while the corrected spelling follows `### Response:`. Keeping the desired response
separate from `model_input` will later make the prompt available for generation without revealing the reference answer.

In [9]:
# Format an example that requires an additional misspelled-word input.
model_input = format_input(data[50])
# Append the supervised continuation only after constructing the prompt.
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


### Formatting a self-contained instruction

When `input` is empty, omit its heading and move directly from the instruction to the response. This avoids teaching the model to expect
an unnecessary blank section.

In [10]:
# Self-contained instructions omit the `### Input:` section entirely.
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


## 7.4 Splitting instruction examples

Use 85% for parameter updates, 5% for validation, and 10% for final testing. Integer rounding assigns any remainder to validation so all
1,100 entries are retained exactly once.

This deterministic contiguous split follows the book and assumes the source ordering is sufficiently mixed. For a newly assembled or
ordered dataset, shuffle reproducibly before slicing or use a stratification strategy appropriate to its task categories.

In [11]:
# Reserve 85% for optimization, 10% for final testing, and the 5% remainder
# for validation-guided development.
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
val_portion = len(data) - train_portion - test_portion

test_start = train_portion
val_start = train_portion + test_portion
# Slices are non-overlapping, so each example belongs to exactly one split.
train_data = data[:train_portion]
test_data = data[test_start:val_start]
val_data = data[val_start:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


## Chapter 7 summary

The chapter begins with a validated instruction-data pipeline:

- cache and validate 1,100 JSON examples from the book repository;
- distinguish instructions, optional task inputs, and desired responses;
- format examples with a consistent Alpaca-style structure; and
- create non-overlapping 85%/5%/10% training, validation, and test splits.

Instruction fine-tuning still uses next-token prediction. Structured supervised examples teach the model that text following the response
header should satisfy the preceding request.